In [ ]:

import csv
import gc
import os
import re
import shutil
import traceback
from collections import defaultdict
from datetime import datetime, timedelta

import mne
import numpy as np


In [ ]:
# ==========================================================================
# 0. CONSTANTS AND PATHS -- STORAGE-EFFICIENT, GAP-AWARE SEGMENTATION
# ==========================================================================
EXPECTED_FS = 256.0
SAMPLE_PERIOD_SEC = 1.0 / EXPECTED_FS
PREICTAL_SEC = 30 * 60
INTERICTAL_DIST_SEC = 4 * 60 * 60
POSTICTAL_SEC = 60 * 60
CONTINUOUS_TOLERANCE_SEC = SAMPLE_PERIOD_SEC + 1e-9
MAX_SHORT_GAP_SEC = 10.0
EPOCH_LEN_SEC = 10

EXPECTED_PREICTAL_EPOCHS = PREICTAL_SEC // EPOCH_LEN_SEC
INTERICTAL_POOL_MULTIPLIER = 3
MIN_INTERICTAL_EPOCHS_PER_SEIZURE = EXPECTED_PREICTAL_EPOCHS * INTERICTAL_POOL_MULTIPLIER
MIN_ELIGIBLE_SEIZURES_PER_PATIENT = 3
BATCH_SIZE = 1800
AFTER_LAST_SEIZURE = 0
PREICTAL_MAX_LOOKBACK_SEC = 35 * 60
INTERICTAL_ASSIGNMENT_CHUNK_SEC = 30 * 60

# Storage controls. EEG is quantized at 0.1 microvolt/count; downstream restores
# volts by multiplying int16 arrays by EEG_INT16_SCALE_VOLTS.
STORAGE_DTYPE = np.int16
EEG_INT16_SCALE_VOLTS = 1e-7
MAX_INTERICTAL_EPOCHS_PER_SEIZURE = MIN_INTERICTAL_EPOCHS_PER_SEIZURE
MIN_FREE_DISK_GIB = 1.0
RESET_OUTPUT_ROOT = True
PATIENTS_PER_BATCH = 3
BATCH_NUMBER = 0

# Kaggle currently exposes more than one input-mount layout.  Locate the
# actual CHB-MIT root by its required chb01 EDF and summary files instead of
# hard-coding the owner/slug prefix.
from pathlib import Path

def discover_chbmit_root(input_root='/kaggle/input'):
    matches = []
    for chb01_dir in Path(input_root).rglob('chb01'):
        if (chb01_dir.is_dir()
                and any(chb01_dir.glob('*.edf'))
                and any(chb01_dir.glob('*summary.txt'))):
            matches.append(chb01_dir.parent)
    matches = sorted(set(matches), key=lambda p: (len(p.parts), str(p)))
    if not matches:
        top = sorted(str(p) for p in Path(input_root).iterdir())
        raise FileNotFoundError(
            'Could not locate raw CHB-MIT data under /kaggle/input. '
            f'Attached top-level inputs: {top}. Add the raw CHB-MIT dataset first.')
    if len(matches) > 1:
        print('[WARN] Multiple CHB-MIT roots found; using:', matches[0])
        print('Other candidates:', matches[1:])
    return str(matches[0])

LOCAL_DATA = discover_chbmit_root()
print('Detected CHB-MIT root:', LOCAL_DATA)
SAVE_ROOT = '/kaggle/working/CHB-MIT_Segmentation_GAPAWARE_V3'

CHANNELS_18 = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ',
]


In [ ]:

# ==========================================================================
# 1. CHANNEL SELECTION AND SEIZURE ANNOTATIONS
# ==========================================================================
def select_channels(raw, channel_list=CHANNELS_18):
    """Require all channels and force one invariant channel order.

    A recording with a missing required channel is excluded; it is never allowed
    to produce a smaller matrix with shifted channel indices.
    """
    if 'T8-P8-0' in raw.ch_names:
        raw.rename_channels({'T8-P8-0': 'T8-P8'})
    raw.rename_channels({ch: ch.strip().replace('EEG ', '').upper()
                         for ch in raw.ch_names})

    missing = [ch for ch in channel_list if ch not in raw.ch_names]
    if missing:
        raise ValueError(f'missing required channels: {missing}')

    raw.pick(channel_list)
    raw.reorder_channels(channel_list)
    if raw.ch_names != list(channel_list):
        raise AssertionError('channel order is not invariant')
    return raw


def load_seizures_from_summary(summary_path, edf_filename):
    seizures = []
    if not os.path.exists(summary_path):
        return seizures
    with open(summary_path, 'r', encoding='latin-1') as f:
        content = f.read()
    for file_block in content.split('File Name:'):
        if edf_filename.lower() not in file_block.lower():
            continue
        n_match = re.search(r'Number of Seizures in File:\s*(\d+)', file_block)
        if not n_match or int(n_match.group(1)) == 0:
            return seizures
        onsets = re.findall(r'Seizure\s*(?:\d+)?\s*Start Time:\s*(\d+)\s*seconds', file_block)
        offsets = re.findall(r'Seizure\s*(?:\d+)?\s*End Time:\s*(\d+)\s*seconds', file_block)
        seizures = [(float(on), float(off)) for on, off in zip(onsets, offsets)]
        break
    return seizures


In [ ]:
# ==========================================================================
# 1.1 QUICK CHECK: NUMBER OF SEIZURES PER PATIENT
# ==========================================================================
seizure_counts = defaultdict(int)

for folder in sorted(os.listdir(LOCAL_DATA)):
    folder_dir = os.path.join(LOCAL_DATA, folder)
    if not os.path.isdir(folder_dir):
        continue

    # Each patient folder has one summary file
    for filename in os.listdir(folder_dir):
        if not filename.endswith('-summary.txt'):
            continue
        with open(os.path.join(folder_dir, filename), 'r') as f:
            matches = re.findall(r'Number of Seizures in File:\s*(\d+)', f.read())
        seizure_counts[folder.upper()] = sum(int(x) for x in matches)
        break

for patient, n in sorted(seizure_counts.items()):
    print(f'{patient}: {n} seizures')

In [ ]:
# ==========================================================================
# 2. BUILD THE RECORDING TIMELINE OF ONE PATIENT
# ==========================================================================
def _discover_edf_files(subject_dir):
    return sorted(f for f in os.listdir(subject_dir) if f.lower().endswith('.edf'))


def _read_file_infos(subject_dir, subject, edf_files):
    """Read every EDF header and attach seizure annotations."""
    summary_path = os.path.join(subject_dir, f'{subject}-summary.txt')
    file_infos = []
    for f in edf_files:
        raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
        try:
            raw = select_channels(raw, CHANNELS_18)
        except ValueError as exc:
            print(f'[EXCLUDE EDF] {f}: {exc}')
            continue
        if not np.isclose(float(raw.info['sfreq']), EXPECTED_FS):
            print(f"[EXCLUDE EDF] {f}: sfreq={raw.info['sfreq']} Hz, expected {EXPECTED_FS} Hz")
            continue
        meas_date = raw.info['meas_date']
        if meas_date is None:
            print(f'[WARN] {f}: no meas_date -> skipping file')
            continue
        duration = raw.times[-1] + 1.0 / raw.info['sfreq']
        file_infos.append({
            'file': f, 'raw': raw, 'start': meas_date, 'duration': duration,
            'end': meas_date + timedelta(seconds=duration),
            'seizures': load_seizures_from_summary(summary_path, f),
        })
    file_infos.sort(key=lambda x: x['start'])
    return file_infos


def _group_into_blocks(file_infos):
    """Keep gaps <=10 s in one logical block; split larger gaps and overlaps."""
    if not file_infos:
        return []
    blocks = [[file_infos[0]]]
    for prev, cur in zip(file_infos, file_infos[1:]):
        gap = (cur['start'] - prev['end']).total_seconds()
        if -CONTINUOUS_TOLERANCE_SEC <= gap <= MAX_SHORT_GAP_SEC:
            if gap > CONTINUOUS_TOLERANCE_SEC:
                print(f'[TOLERATED SHORT GAP] {prev["file"]} -> {cur["file"]}: {gap:.3f} s')
            blocks[-1].append(cur)
        else:
            kind = 'OVERLAP' if gap < 0 else 'GAP'
            print(f'[SPLIT BLOCK: {kind}] {prev["file"]} -> {cur["file"]}: {gap:.3f} s')
            blocks.append([cur])
    return blocks


def _concat_block(block_files):
    """Concatenate EDF samples while retaining MNE boundary annotations."""
    raws, data_cursor = [], 0.0
    for fi in block_files:
        fi['data_start'] = data_cursor
        raw = fi['raw'].copy().load_data()
        raws.append(raw)
        data_cursor += fi['duration']
    raw_concat = raws[0] if len(raws) == 1 else mne.concatenate_raws(raws)
    # BAD/EDGE boundary annotations deliberately remain: reject_by_annotation=True
    # then prevents a 10-s epoch from containing samples from two EDF files.
    raw_concat.notch_filter(freqs=60.0, verbose=False)
    return raw_concat, block_files[0]['start']


def data_sec_to_abs(block, sec):
    """Map compressed concatenated-data seconds to the original EDF clock."""
    eps = SAMPLE_PERIOD_SEC + 1e-9
    for idx, fi in enumerate(block['files']):
        lo, hi = fi['data_start'], fi['data_start'] + fi['duration']
        is_last = idx == len(block['files']) - 1
        # Concatenation boundaries belong to the following EDF, not the
        # preceding EDF's wall-clock end (which may be separated by a gap).
        if lo - eps <= sec < hi or (is_last and sec <= hi + eps):
            within = min(max(sec - lo, 0.0), fi['duration'])
            return fi['start'] + timedelta(seconds=within)
    raise ValueError(f'data second {sec} is outside block')


def abs_to_data_sec(block, when):
    """Map an EDF-clock datetime to compressed concatenated-data seconds."""
    for fi in block['files']:
        if fi['start'] <= when <= fi['end']:
            return fi['data_start'] + (when - fi['start']).total_seconds()
    # A requested interval endpoint may fall inside a tolerated gap. Map it to
    # the adjacent sample boundary; the missing wall-clock time is never invented.
    for prev, cur in zip(block['files'], block['files'][1:]):
        if prev['end'] < when < cur['start']:
            return cur['data_start']
    raise ValueError(f'absolute time {when} is outside recorded EDF data')


def build_timeline(subject_dir, subject):
    edf_files = _discover_edf_files(subject_dir)
    file_infos = _read_file_infos(subject_dir, subject, edf_files)
    blocks = []
    for block_files in _group_into_blocks(file_infos):
        raw_concat, block_start = _concat_block(block_files)
        blocks.append({
            'raw': raw_concat, 'start': block_start,
            'duration': raw_concat.times[-1] + 1.0 / raw_concat.info['sfreq'],
            'end': block_files[-1]['end'], 'files': block_files,
        })
    blocks.sort(key=lambda b: b['start'])
    return blocks


In [ ]:
# ==========================================================================
# 3. COLLECT ALL SEIZURES OF A PATIENT
# ==========================================================================
def collect_seizures(blocks):
    """Return seizures with compressed-data and absolute EDF-clock coordinates."""
    all_seizures = []
    for b_idx, block in enumerate(blocks):
        for finfo in block['files']:
            for onset, offset in finfo['seizures']:
                all_seizures.append({
                    'block_idx': b_idx,
                    'onset_in_block': finfo['data_start'] + onset,
                    'offset_in_block': finfo['data_start'] + offset,
                    'onset_abs': finfo['start'] + timedelta(seconds=onset),
                    'offset_abs': finfo['start'] + timedelta(seconds=offset),
                    'source_file': finfo['file'],
                })
    all_seizures.sort(key=lambda s: s['onset_abs'])
    for i, s in enumerate(all_seizures):
        s['seizure_id'] = i + 1
    return all_seizures


In [ ]:
# ==========================================================================
# 4. GAP-AWARE PREICTAL DEFINITION
# ==========================================================================
def get_previous_seizure(seizure, all_seizures):
    earlier = [s for s in all_seizures if s['onset_abs'] < seizure['onset_abs']]
    return max(earlier, key=lambda s: s['onset_abs']) if earlier else None


def get_preictal_window(seizure, block, all_seizures):
    """Return a candidate window containing up to 35 min before onset.

    The final extractor retains the latest 180 valid 10-s epochs. This permits
    short EDF gaps while still requiring exactly 30 min of available EEG.
    """
    onset = seizure['onset_in_block']
    if onset < PREICTAL_SEC:
        return None
    return max(0.0, onset - PREICTAL_MAX_LOOKBACK_SEC), onset


In [ ]:
# ==========================================================================
# 5. INTERICTAL WINDOWS AND NON-OVERLAPPING LOSO ALLOCATION
# ==========================================================================
def get_interictal_windows(blocks, all_seizures):
    """Return valid interictal windows separately inside each physical EDF file."""
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    margin = timedelta(seconds=INTERICTAL_DIST_SEC)
    windows = []
    for b_idx, block in enumerate(blocks):
        for fi in block['files']:
            lo, hi = fi['start'], fi['end']
            excluded = []
            for s in seizures:
                e0, e1 = max(s['onset_abs'] - margin, lo), min(s['offset_abs'] + margin, hi)
                if e0 < e1:
                    excluded.append((e0, e1))
            excluded.sort()
            merged = []
            for e0, e1 in excluded:
                if merged and e0 <= merged[-1][1]:
                    merged[-1] = (merged[-1][0], max(merged[-1][1], e1))
                else:
                    merged.append((e0, e1))
            cursor = lo
            for e0, e1 in merged:
                if cursor < e0:
                    windows.append((cursor, e0, b_idx))
                cursor = max(cursor, e1)
            if cursor < hi:
                windows.append((cursor, hi, b_idx))
    return sorted(windows, key=lambda w: w[0])


def split_interictal_windows(windows, chunk_sec=INTERICTAL_ASSIGNMENT_CHUNK_SEC):
    """Split the global pool into deterministic, non-overlapping chunks."""
    pieces = []
    for start, end, b_idx in windows:
        cursor = start
        while (end - cursor).total_seconds() >= EPOCH_LEN_SEC:
            piece_end = min(end, cursor + timedelta(seconds=chunk_sec))
            pieces.append((cursor, piece_end, b_idx))
            cursor = piece_end
    return pieces


def assign_windows_to_seizures(windows, seizure_ids):
    """Greedily balance disjoint interictal chunks among candidate seizures."""
    ids = sorted(seizure_ids)
    assigned = {sid: [] for sid in ids}
    capacity = {sid: 0 for sid in ids}
    for piece in split_interictal_windows(windows):
        if not ids:
            break
        sid = min(ids, key=lambda x: (capacity[x], x))
        assigned[sid].append(piece)
        capacity[sid] += window_epoch_capacity([piece])
    return assigned


In [ ]:
# ==========================================================================
# 5.1 ELIGIBILITY AUDIT
# ==========================================================================
def window_epoch_capacity(windows):
    return int(sum(max(0.0, (end - start).total_seconds()) // EPOCH_LEN_SEC
                   for start, end, _ in windows))


def audit_eligible_seizures(blocks, all_seizures):
    preictal_ok = {}
    for seizure in all_seizures:
        epochs = get_available_preictal_epochs(
            blocks[seizure['block_idx']], seizure, all_seizures)
        preictal_ok[seizure['seizure_id']] = epochs is not None
        if epochs is not None:
            del epochs
    gc.collect()

    candidate_ids = [sid for sid, ok in preictal_ok.items() if ok]
    inter_windows = get_interictal_windows(blocks, all_seizures)
    assigned = assign_windows_to_seizures(inter_windows, candidate_ids)

    rows, eligible_ids = [], []
    for seizure in all_seizures:
        sid = seizure['seizure_id']
        full_preictal = preictal_ok[sid]
        n_interictal = window_epoch_capacity(assigned.get(sid, []))
        adequate_interictal = n_interictal >= MIN_INTERICTAL_EPOCHS_PER_SEIZURE
        eligible = full_preictal and adequate_interictal
        if eligible:
            eligible_ids.append(sid)
        rows.append({
            'SeizureID': sid, 'Onset': seizure['onset_abs'].isoformat(),
            'Offset': seizure['offset_abs'].isoformat(),
            'Full_30min_Available_Preictal': int(full_preictal),
            'Expected_Preictal_Epochs': EXPECTED_PREICTAL_EPOCHS if full_preictal else 0,
            'Allocated_Interictal_EpochCapacity': n_interictal,
            'Adequate_Interictal': int(adequate_interictal), 'Eligible': int(eligible),
            'Reason': ('eligible' if eligible else
                       'fewer_than_180_valid_preictal_epochs' if not full_preictal else
                       'insufficient_allocated_interictal'),
        })
    return rows, eligible_ids


def write_qc_csv(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not rows:
        return
    # Cohort rows may contain an additional Error field only for failed subjects.
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
# ==========================================================================
# 6. TESTING SEGMENTS (CONTINUOUS DATA BETWEEN SEIZURES, FOR FAR/h)
# ==========================================================================
def get_testing_segments(blocks, all_seizures, include_tail=False):
    """Return monitoring segments using compressed-data coordinates per block."""
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    rec_start = min(b['start'] for b in blocks)
    rec_end = max(b['end'] for b in blocks)
    spans, prev_end = [], rec_start
    for s in seizures:
        if prev_end < s['onset_abs']:
            spans.append((s['seizure_id'], prev_end, s['onset_abs']))
        prev_end = max(prev_end, s['offset_abs'])
    if include_tail and prev_end < rec_end:
        spans.append(('tail', prev_end, rec_end))

    result = {}
    for name, lo, hi in spans:
        segments = []
        for b_idx, block in enumerate(blocks):
            o0, o1 = max(lo, block['start']), min(hi, block['end'])
            if o0 >= o1:
                continue
            try:
                w0, w1 = abs_to_data_sec(block, o0), abs_to_data_sec(block, o1)
            except ValueError:
                continue
            if w1 - w0 >= EPOCH_LEN_SEC:
                segments.append((b_idx, w0, w1))
        if segments:
            result[name] = segments
    return result


In [ ]:
# ==========================================================================
# 7. EPOCHING, PHASE LABELS AND SAVING
# ==========================================================================
def save_fold_npy(data, name, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    free_gb = shutil.disk_usage(out_dir).free / (1024 ** 3)
    if free_gb < MIN_FREE_DISK_GIB:
        raise OSError(f'Only {free_gb:.2f} GiB free; refusing to fill the Kaggle disk')
    out_path = os.path.join(out_dir, f'{name}.npy')
    data = np.asarray(data)
    scaled = np.rint(data / EEG_INT16_SCALE_VOLTS)
    info = np.iinfo(STORAGE_DTYPE)
    out_of_range = int(np.count_nonzero((scaled < info.min) | (scaled > info.max)))
    # Preserve rare high-amplitude samples losslessly. Most files remain int16;
    # only a file containing an out-of-range value falls back to int32.
    if out_of_range:
        compact = scaled.astype(np.int32)
        storage_label = f'int32 fallback ({out_of_range} samples exceed int16)'
    else:
        compact = scaled.astype(STORAGE_DTYPE)
        storage_label = 'int16'
    np.save(out_path, compact, allow_pickle=False)
    print(f'Saved {out_path} | {storage_label} @ 0.1 uV/count | '
          f'{compact.nbytes / 2**20:.1f} MiB | free {free_gb:.2f} GiB')
    del scaled, compact


def crop_to_epochs(raw, t_start, t_end, epoch_len=EPOCH_LEN_SEC, expected_epochs=None):
    sample_period = 1.0 / float(raw.info['sfreq'])
    # raw.times[-1] is the time of the final sample, whereas interval endpoints
    # are exclusive. Add one sample so a complete final 10-s epoch is not lost.
    available_end = raw.times[-1] + sample_period
    t_end = min(t_end, available_end)
    if t_end - t_start < epoch_len:
        return None
    cropped = raw.copy().crop(
        tmin=max(t_start, 0), tmax=min(t_end - sample_period, raw.times[-1]),
        include_tmax=True)
    epochs = mne.make_fixed_length_epochs(
        cropped, duration=epoch_len, reject_by_annotation=True, preload=True, verbose=False)
    del cropped
    epochs.drop_bad()
    if len(epochs) == 0:
        del epochs
        gc.collect()
        return None
    if expected_epochs is not None and len(epochs) != expected_epochs:
        print(f'[EXCLUDE WINDOW] expected {expected_epochs} epochs, obtained {len(epochs)}')
        del epochs
        gc.collect()
        return None
    if epochs.get_data(copy=False).shape[1] != len(CHANNELS_18):
        raise AssertionError('epoch data do not contain exactly 18 channels')
    return epochs


def epoch_start_seconds(epochs, reference_raw):
    return (epochs.events[:, 0] - reference_raw.first_samp) / reference_raw.info['sfreq']


def _epoch_overlaps(epoch_start_abs, epoch_end_abs, lo, hi):
    return epoch_start_abs < hi and epoch_end_abs > lo


def get_available_preictal_epochs(block, seizure, all_seizures):
    """Return the latest exactly 180 valid epochs within a 35-min lookback."""
    window = get_preictal_window(seizure, block, all_seizures)
    if window is None:
        return None
    epochs = crop_to_epochs(block['raw'], *window)
    if epochs is None:
        return None
    valid = []
    for idx, sec in enumerate(epoch_start_seconds(epochs, block['raw'])):
        a0 = data_sec_to_abs(block, float(sec))
        a1 = a0 + timedelta(seconds=EPOCH_LEN_SEC)
        if a1 > seizure['onset_abs']:
            continue
        contaminated = False
        for other in all_seizures:
            if other['seizure_id'] != seizure['seizure_id'] and _epoch_overlaps(
                    a0, a1, other['onset_abs'],
                    other['offset_abs'] + timedelta(seconds=POSTICTAL_SEC)):
                contaminated = True
                break
        if not contaminated:
            valid.append(idx)
    if len(valid) < EXPECTED_PREICTAL_EPOCHS:
        del epochs
        gc.collect()
        return None
    selected = epochs[valid[-EXPECTED_PREICTAL_EPOCHS:]]
    del epochs
    return selected


def save_epochs_in_batches(epochs, out_dir, start_batch_idx=0, batch_size=BATCH_SIZE, tag=''):
    n, batch_idx = len(epochs), start_batch_idx
    for i in range(0, n, batch_size):
        print(f'{tag} | Epochs: {n} | Batch: {i}:{min(i + batch_size, n)}')
        batch = epochs[i:i + batch_size].get_data(copy=False)
        save_fold_npy(batch, batch_idx, out_dir)
        batch_idx += 1
        del batch
    return batch_idx


def classify_epoch_phase(epoch_start_abs, epoch_end_abs, target_seizure, all_seizures):
    for s in all_seizures:
        if _epoch_overlaps(epoch_start_abs, epoch_end_abs, s['onset_abs'], s['offset_abs']):
            return 'ictal'
    for s in all_seizures:
        if _epoch_overlaps(epoch_start_abs, epoch_end_abs, s['offset_abs'],
                           s['offset_abs'] + timedelta(seconds=POSTICTAL_SEC)):
            return 'postictal'
    margin = timedelta(seconds=INTERICTAL_DIST_SEC)
    fully_interictal = all(
        epoch_end_abs <= s['onset_abs'] - margin or
        epoch_start_abs >= s['offset_abs'] + margin for s in all_seizures)
    return 'interictal' if fully_interictal else 'excluded'


In [ ]:
# ==========================================================================
# 8. PROCESSING PIPELINE FOR ONE PATIENT
# ==========================================================================
def extract_preictal(blocks, all_seizures, out_dir, eligible_ids):
    total = 0
    for s in all_seizures:
        sid = s['seizure_id']
        if sid not in eligible_ids:
            continue
        epochs = get_available_preictal_epochs(blocks[s['block_idx']], s, all_seizures)
        if epochs is None:
            raise RuntimeError(f'Audited seizure {sid} lost its valid preictal epochs')
        save_fold_npy(epochs.get_data(copy=False), sid, out_dir)
        total += len(epochs)
        del epochs
        print(f'  Seizure {sid:02d}: 180 valid preictal epochs OK')
    print(f'Total preictal epochs: {total}')
    return total


def extract_interictal(subject, blocks, all_seizures, out_root, eligible_ids):
    """Save 540 temporally distributed interictal epochs per eligible seizure.

    A two-pass procedure measures the actual valid epoch count first and then
    selects evenly spaced indices across all assigned windows. This avoids a
    temporal bias toward the first EDF files without loading the whole pool at once.
    """
    windows = get_interictal_windows(blocks, all_seizures)
    windows_by_seizure = assign_windows_to_seizures(windows, eligible_ids)
    total = 0
    for sz_idx in sorted(eligible_ids):
        out_dir = os.path.join(out_root, f'seizure_{sz_idx:02d}')
        os.makedirs(out_dir, exist_ok=True)
        valid_windows = []

        # Pass 1: obtain actual valid counts (rather than relying only on duration).
        for a0, a1, b_idx in windows_by_seizure.get(sz_idx, []):
            block = blocks[b_idx]
            w0, w1 = abs_to_data_sec(block, a0), abs_to_data_sec(block, a1)
            epochs = crop_to_epochs(block['raw'], w0, w1)
            if epochs is not None:
                valid_windows.append((a0, a1, b_idx, len(epochs)))
                del epochs

        pool_size = sum(item[3] for item in valid_windows)
        if pool_size < MAX_INTERICTAL_EPOCHS_PER_SEIZURE:
            raise RuntimeError(
                f'Seizure {sz_idx}: only {pool_size}/'
                f'{MAX_INTERICTAL_EPOCHS_PER_SEIZURE} valid interictal epochs')

        # Deterministic uniform coverage of the complete chronological pool.
        selected_global = np.linspace(
            0, pool_size - 1, MAX_INTERICTAL_EPOCHS_PER_SEIZURE,
            dtype=np.int64)
        selected_chunks, cursor = [], 0
        for a0, a1, b_idx, n_epochs in valid_windows:
            mask = (selected_global >= cursor) & (selected_global < cursor + n_epochs)
            local_indices = selected_global[mask] - cursor
            cursor += n_epochs
            if len(local_indices) == 0:
                continue
            block = blocks[b_idx]
            w0, w1 = abs_to_data_sec(block, a0), abs_to_data_sec(block, a1)
            epochs = crop_to_epochs(block['raw'], w0, w1)
            if epochs is None or len(epochs) != n_epochs:
                raise RuntimeError('Interictal epoch count changed between selection passes')
            selected_chunks.append(
                epochs.get_data(copy=False)[local_indices].astype(np.float32, copy=True))
            del epochs

        selected_data = np.concatenate(selected_chunks, axis=0)
        if len(selected_data) != MAX_INTERICTAL_EPOCHS_PER_SEIZURE:
            raise AssertionError('Uniform interictal selection did not return exactly 540 epochs')
        save_fold_npy(selected_data, 0, out_dir)
        total += len(selected_data)
        print(f'  Seizure {sz_idx:02d}: selected {len(selected_data)}/{pool_size} '
              'temporally distributed interictal epochs')
        del selected_data, selected_chunks
        gc.collect()
    print(f'Total interictal epochs: {total}')
    return total


def extract_testing(subject, blocks, all_seizures, out_root, eligible_ids):
    """Save continuous streams and an epoch-level, gap-aware phase manifest."""
    total, manifest = 0, []
    by_id = {s['seizure_id']: s for s in all_seizures}
    for name, segments in get_testing_segments(blocks, all_seizures).items():
        # LOSO folds exist only for seizures that passed both audits. All other
        # seizures remain in all_seizures so their contaminated time is excluded.
        if not isinstance(name, int) or name not in by_id or name not in eligible_ids:
            continue
        target, row_start = by_id[name], len(manifest)
        tag, global_epoch = f'seizure_{name:02d}', 0
        print(f'Testing: {tag} | {len(segments)} segments')
        for segment_idx, (b_idx, w0, w1) in enumerate(segments):
            block = blocks[b_idx]
            out_dir = os.path.join(out_root, tag, f'block_{b_idx:02d}')
            os.makedirs(out_dir, exist_ok=True)
            epochs = crop_to_epochs(block['raw'], w0, w1)
            if epochs is None:
                continue
            starts, data = epoch_start_seconds(epochs, block['raw']), epochs.get_data(copy=False)
            for batch_idx, i in enumerate(range(0, len(epochs), BATCH_SIZE)):
                j = min(i + BATCH_SIZE, len(epochs))
                save_fold_npy(data[i:j], batch_idx, out_dir)
                rel = os.path.relpath(os.path.join(out_dir, f'{batch_idx}.npy'), out_root)
                for local_idx, start_sec in enumerate(starts[i:j]):
                    a0 = data_sec_to_abs(block, float(start_sec))
                    a1 = a0 + timedelta(seconds=EPOCH_LEN_SEC)
                    phase = classify_epoch_phase(a0, a1, target, all_seizures)
                    manifest.append({
                        'Subject': subject, 'SeizureID': name, 'Block': b_idx,
                        'NpyRelativePath': rel.replace(os.sep, '/'),
                        'EpochInFile': local_idx, 'Epoch': global_epoch,
                        'EpochStart': a0.isoformat(), 'Phase': phase,
                        'TargetPreictal': 0, 'EligibleForFAR': int(phase == 'interictal'),
                        'Discontinuity': 0,
                    })
                    global_epoch += 1
                total += j - i
            del data, epochs
            gc.collect()

        # Mark the latest 180 clean epochs within the same 35-min definition used
        # for training. Any EDF gap is visible as a jump in EpochStart.
        rows = manifest[row_start:]
        lookback = target['onset_abs'] - timedelta(seconds=PREICTAL_MAX_LOOKBACK_SEC)
        candidates = []
        for idx, row in enumerate(rows):
            a0 = datetime.fromisoformat(row['EpochStart'])
            if a0 >= lookback and row['Phase'] not in {'ictal', 'postictal'}:
                candidates.append(idx)
        selected = set(candidates[-EXPECTED_PREICTAL_EPOCHS:])
        for idx, row in enumerate(rows):
            if idx in selected:
                row['Phase'], row['TargetPreictal'], row['EligibleForFAR'] = 'preictal', 1, 0
            if idx > 0:
                prev = datetime.fromisoformat(rows[idx - 1]['EpochStart'])
                cur = datetime.fromisoformat(row['EpochStart'])
                if (cur - prev).total_seconds() > EPOCH_LEN_SEC + CONTINUOUS_TOLERANCE_SEC:
                    row['Discontinuity'] = 1

    write_qc_csv(os.path.join(os.path.dirname(out_root), 'testing_epoch_manifest.csv'), manifest)
    print(f'Total testing epochs: {total}')
    return total



def process_subject(subject, phase=(1, 2, 3)):
    phases = {phase} if isinstance(phase, int) else set(phase)
    subject_dir = os.path.join(LOCAL_DATA, subject)
    save_dir = os.path.join(SAVE_ROOT, subject)
    training_dir = os.path.join(save_dir, 'Training')

    print(f'\n--- PROCESSING: {subject.upper()} ---')
    blocks = build_timeline(subject_dir, subject)
    if not blocks:
        print(f'{subject}: no valid 18-channel continuous blocks, excluded')
        return {'Subject': subject, 'Included': 0, 'EligibleSeizures': 0}

    all_seizures = collect_seizures(blocks)
    if not all_seizures:
        print(f'{subject}: no valid seizures, excluded')
        return {'Subject': subject, 'Included': 0, 'EligibleSeizures': 0}

    qc_rows, eligible_ids = audit_eligible_seizures(blocks, all_seizures)
    write_qc_csv(os.path.join(save_dir, 'segmentation_qc.csv'), qc_rows)

    included = len(eligible_ids) >= MIN_ELIGIBLE_SEIZURES_PER_PATIENT
    print(f'{subject}: {len(eligible_ids)}/{len(all_seizures)} eligible seizures')
    if not included:
        print(f'[EXCLUDE PATIENT] fewer than {MIN_ELIGIBLE_SEIZURES_PER_PATIENT} eligible seizures')
        return {'Subject': subject, 'Included': 0, 'EligibleSeizures': len(eligible_ids)}

    if 1 in phases:
        extract_preictal(blocks, all_seizures,
                         os.path.join(training_dir, 'preictal'), eligible_ids)
    if 2 in phases:
        extract_interictal(subject, blocks, all_seizures,
                           os.path.join(training_dir, 'interictal'), eligible_ids)
    if 3 in phases:
        # Continuous monitoring data are kept separate from eligible interictal
        # training data. Downstream code must not label every non-preictal epoch
        # in this stream as interictal.
        extract_testing(subject, blocks, all_seizures,
                        os.path.join(save_dir, 'testing'), eligible_ids)

    return {'Subject': subject, 'Included': 1, 'EligibleSeizures': len(eligible_ids)}


In [ ]:

# ==========================================================================
# 9. PROCESS ALL PATIENTS AND SAVE THE FINAL COHORT MANIFEST
# ==========================================================================
START_FROM = 'chb01'
STOP_AT = 'chb01'

def discover_subjects(local_data, start_from=None, stop_at=None):
    pattern = re.compile(r'^chb\d{2}$')
    subjects = sorted(d for d in os.listdir(local_data)
                      if pattern.match(d) and os.path.isdir(os.path.join(local_data, d)))
    if start_from is not None:
        subjects = [s for s in subjects if s >= start_from.lower()]
    if stop_at is not None:
        subjects = [s for s in subjects if s <= stop_at.lower()]
    return subjects


if __name__ == '__main__':
    if RESET_OUTPUT_ROOT and os.path.isdir(SAVE_ROOT):
        print(f'[RESET OUTPUT] Removing previous run: {SAVE_ROOT}')
        shutil.rmtree(SAVE_ROOT)
    os.makedirs(SAVE_ROOT, exist_ok=True)
    subjects = discover_subjects(LOCAL_DATA, START_FROM, STOP_AT)
    if START_FROM is None and STOP_AT is None and PATIENTS_PER_BATCH is not None:
        batch_start = BATCH_NUMBER * PATIENTS_PER_BATCH
        subjects = subjects[batch_start:batch_start + PATIENTS_PER_BATCH]
        print(f'Batch {BATCH_NUMBER}: {subjects}')
    print(f'Found {len(subjects)} patient folders: {subjects}')
    cohort_rows = []
    for subject in subjects:
        try:
            cohort_rows.append(process_subject(subject, phase=(1, 2, 3)))
        except Exception as exc:
            print(f'[ERROR] {subject}: {exc}')
            traceback.print_exc()
            cohort_rows.append({'Subject': subject, 'Included': 0,
                                'EligibleSeizures': 0, 'Error': repr(exc)})
    write_qc_csv(os.path.join(SAVE_ROOT, 'cohort_manifest.csv'), cohort_rows)


In [ ]:
# ==========================================================================
# 10. AUTOMATIC QC SUMMARY FOR THE PILOT RUN
# ==========================================================================
import pandas as pd
from pathlib import Path


def count_saved_epochs(path):
    total = 0
    for npy_path in Path(path).rglob('*.npy'):
        total += int(np.load(npy_path, mmap_mode='r').shape[0])
    return total


cohort_path = Path(SAVE_ROOT) / 'cohort_manifest.csv'
if not cohort_path.exists():
    raise FileNotFoundError(f'Missing {cohort_path}; inspect the processing log above')

cohort = pd.read_csv(cohort_path)
display(cohort)

for subject in cohort['Subject']:
    subject_dir = Path(SAVE_ROOT) / subject
    qc_path = subject_dir / 'segmentation_qc.csv'
    testing_path = subject_dir / 'testing_epoch_manifest.csv'
    print(f'\n===== {subject} =====')
    if qc_path.exists():
        qc = pd.read_csv(qc_path)
        display(qc)
        print('Eligible seizures:', int(qc['Eligible'].sum()), '/', len(qc))
    else:
        print('[MISSING]', qc_path)

    cohort_row = cohort.loc[cohort['Subject'] == subject].iloc[0]
    eligible_n = int(cohort_row['EligibleSeizures'])
    included = int(cohort_row['Included']) == 1
    # Excluded patients deliberately have no saved training/testing data even
    # when one or two individual seizures happen to pass the seizure audit.
    expected_train_pre = (eligible_n * EXPECTED_PREICTAL_EPOCHS) if included else 0
    expected_train_inter = (
        eligible_n * MAX_INTERICTAL_EPOCHS_PER_SEIZURE) if included else 0
    actual_train_pre = count_saved_epochs(subject_dir / 'Training' / 'preictal')
    actual_train_inter = count_saved_epochs(subject_dir / 'Training' / 'interictal')
    print(f'Training preictal: {actual_train_pre}/{expected_train_pre}',
          'PASS' if actual_train_pre == expected_train_pre else 'FAIL')
    print(f'Training interictal: {actual_train_inter}/{expected_train_inter}',
          'PASS' if actual_train_inter == expected_train_inter else 'FAIL')

    if testing_path.exists():
        testing = pd.read_csv(testing_path)
        print('Testing epochs:', len(testing))
        display(testing['Phase'].value_counts(dropna=False).rename('Epochs').to_frame())
        print('Discontinuities:', int(testing['Discontinuity'].sum()))
        print('Duplicate epoch keys:', int(testing.duplicated(
            ['SeizureID', 'Block', 'NpyRelativePath', 'EpochInFile']).sum()))
        expected_pre = eligible_n * EXPECTED_PREICTAL_EPOCHS
        actual_pre = int(testing['TargetPreictal'].sum())
        print(f'Preictal consistency: {actual_pre}/{expected_pre}',
              'PASS' if actual_pre == expected_pre else 'FAIL')
    else:
        if included:
            print('[MISSING]', testing_path)
        else:
            print('[EXPECTED] no testing manifest: patient excluded at cohort level')

if START_FROM is not None or STOP_AT is not None:
    print(f'\nRange mode: START_FROM={START_FROM}, STOP_AT={STOP_AT}')
else:
    print(f'\nBatch mode: BATCH_NUMBER={BATCH_NUMBER}, '
          f'PATIENTS_PER_BATCH={PATIENTS_PER_BATCH}')
print('Save this batch before running the next batch.')
